### Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import seaborn as sns
import matplotlib as mpl
import re
import statsmodels.api as sm

from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve, classification_report, auc, brier_score_loss,ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from xgboost import XGBRegressor
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.shared import Inches
from docx.enum.section import WD_ORIENT

# Figures information
mpl.rcParams.update({
    'font.size': 8,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})
mm_to_in = lambda mm: mm / 25.4
fig_width  = mm_to_in(85)
fig_height = mm_to_in(65)
fig_width_large  = mm_to_in(170)
fig_height_large = mm_to_in(130)

### Define variables

In [ ]:
ID_COL  = "GenID"
ISS_COL = "iss"
AGE_COL = "age"
LAC_COL = "lactate_value"
MODS_COL ="mods"
BD_COL = "bd_value"
PEN_COL = "penetrating_mech"

N_SPLITS = 5
TEST_SIZE = 0.2
RANDOM_STATE = 8

thresholds = []
sensitivities = []
specificities = []

top_k = 200  # number of proteins to keep

### Preprocess MRC

In [ ]:
clin_path = "./Datasets/clin_mrc.csv"
prot_path = "./Datasets/prot_mrc.csv"

outcome_map = {'Alive': 1, 'Dead': 0, 'Unknown': np.nan}

mrc_clin = pd.read_csv(clin_path, dtype={'GenID': str})
mrc_clin = mrc_clin.assign(
    GenID="MRC_" + mrc_clin['GenID'].astype(str),
    crit=pd.Categorical((mrc_clin['iss'] >= 25).astype(int), categories=[1, 0], ordered=True),
    sex=pd.Categorical(mrc_clin['sex'], categories=['Male', 'Female']),
    outcome_28d=pd.Categorical(
        mrc_clin['outcome_28d'].replace(outcome_map),
        categories=[1, 0]
    ),
    outcome_24h=pd.Categorical(
        mrc_clin['outcome_24h'].replace(outcome_map),
        categories=[1, 0]
    ),
    penetrating_mech=pd.Categorical(mrc_clin['penetrating_mech'], categories=['Blunt','Penetrating'])
)

mrc_prot = pd.read_csv(prot_path, dtype={'GenID': str})
mrc_prot['GenID'] = "MRC_" + mrc_prot['GenID'].astype(str)

mrc = pd.merge(mrc_clin, mrc_prot, on='GenID', how='inner')

### Preprocess SWAT

In [ ]:
swat_clin_path = "./Datasets/clin_swat.csv"
swat_prot_path = "./Datasets/prot_swat.csv"

swat_clin = pd.read_csv(swat_clin_path, dtype={'GenID': str}).copy()

swat_clin = swat_clin.assign(
    sex=pd.Categorical(swat_clin['sex'], categories=['Male', 'Female'], ordered=True),
    outcome_28d=pd.Categorical(
        swat_clin['outcome_28d'].replace(outcome_map),
        categories=[1, 0]
    ),
    penetrating_mech=pd.Categorical(
        swat_clin['penetrating_mech'], 
        categories=['Blunt', 'Penetrating'],
        ordered=True
    ),
    crit=pd.Categorical(
        np.where(swat_clin['iss'] >= 25, 1, 0),
        categories=[1, 0],
        ordered=True
    ),
    GenID="SWAT_" + swat_clin['GenID'].astype(str)
)

# Controls are set to alive at 28 days
swat_clin.loc[swat_clin['site'] == "control", 'outcome_28d'] = 1
swat_clin['outcome_28d'] = pd.Categorical(swat_clin['outcome_28d'], categories=[1, 0])

swat_prot = pd.read_csv(swat_prot_path, dtype={'GenID': str}).copy()
swat_prot['GenID'] = "SWAT_" + swat_prot['GenID'].astype(str)

swat = pd.merge(swat_clin, swat_prot, on="GenID", how="inner")

swat_mask = ((swat[PEN_COL] == "Blunt") | (swat[PEN_COL].isna())) 
swat_blunt = swat[swat_mask]

### Prep data

In [ ]:
mrc_swat_prots = list(set(mrc_prot.columns).intersection(swat_prot.columns))
prot_cols = sorted(list(set(mrc_swat_prots) & set(mrc.columns) & set(swat.columns) - {ID_COL}))

# use only overlapping proteins in MRC and SWAT
X_mrc = mrc[prot_cols].apply(pd.to_numeric, errors="coerce")
X_swat = swat_blunt[prot_cols].apply(pd.to_numeric, errors="coerce")

# Prediction variable
y_mrc = mrc[ISS_COL].astype(int)
y_swat = swat_blunt[ISS_COL].astype(int)

X = X_mrc.copy()
y = y_mrc.values
ids = X_mrc.index

### Define functions

In [ ]:
def iss_to_bins(y):
    """
    Stratify ISS into bins:
    - Mild:    ISS 0–15
    - Moderate: ISS 16–24
    - Critical: ISS >=25
    """
    y = np.asarray(y)
    bins = np.zeros_like(y, dtype=int)
    bins[(y >= 0) & (y <= 15)] = 0
    bins[(y >= 16) & (y <= 24)] = 1
    bins[y >= 25] = 2
    return bins

def make_selector_xgb():
    return XGBRegressor(
        n_estimators=1000,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.5,
        reg_alpha=2,
        reg_lambda=5,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=1,
        importance_type="gain"
    )

def make_final_xgb():
    return XGBRegressor(
        n_estimators=1200,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.7,
        colsample_bytree=0.3,
        reg_alpha=2,
        reg_lambda=5,
        min_child_weight=5,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=1,
        importance_type="gain"
    )

def set_cell_border(cell, **kwargs):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcBorders = OxmlElement('w:tcBorders')
    for edge in ('top', 'bottom'):
        if edge in kwargs:
            tag = f'w:{edge}'
            el = OxmlElement(tag)
            for key, val in kwargs[edge].items():
                el.set(qn(f'w:{key}'), str(val))
            tcBorders.append(el)
    tcPr.append(tcBorders)

def plot_cm(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["ISS < 25", "ISS \u2265 25"]
    )
    disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(title, fontsize=9)
    ax.tick_params(axis='both', labelsize=7, colors="black")
    ax.xaxis.label.set_size(8)
    ax.yaxis.label.set_size(8)
    for text in disp.text_.ravel():
        text.set_fontsize(8)

def predict_probs(X, y):
    X_const = sm.add_constant(X)
    model = sm.Logit(y, X_const).fit(disp=0)
    return model.predict(X_const)

def get_roc_auc(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)
    return fpr, tpr, auc

def summary_row_median_q1_q3(series):
    return f"{series.median():.2f} [{series.quantile(0.25):.2f}, {series.quantile(0.75):.2f}]"

### Split Data into Test/Training Datasets

In [ ]:
bins = iss_to_bins(y)
X_train_df, X_test_df, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=bins
)
train_bins = iss_to_bins(y_train)

### Stage 1: Nested CV with Feature Stability Tracking

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros_like(y_train, dtype=float)
fold_metrics = []
feature_counts = Counter()

all_shap_values = []
all_shap_X = []

fold = 1
for tr_idx, val_idx in skf.split(X_train_df.values, train_bins):
    print(f"Running fold {fold}")
    X_tr_df = X_train_df.iloc[tr_idx].copy()
    X_val_df = X_train_df.iloc[val_idx].copy()
    y_tr = y_train[tr_idx]
    y_val = y_train[val_idx]

    # Feature selection
    selector_xgb = make_selector_xgb()
    selector_xgb.fit(X_tr_df.values, y_tr)

    importances = selector_xgb.feature_importances_
    top_idx = np.argsort(importances, kind="stable")[-top_k:]
    selected_cols = X_train_df.columns[top_idx]

    X_tr_sel = X_tr_df[selected_cols].values
    X_val_sel = X_val_df[selected_cols].values

     # Track feature stability
    feature_counts.update(selected_cols)

    # Scale
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr_sel)
    X_val_scaled = scaler.transform(X_val_sel)

    # Sample weights
    w = np.where(y_tr >= 25, 4,
         np.where(y_tr >= 16, 2, 1))

    # Final XGB model
    final_xgb = make_final_xgb()
    final_xgb.fit(X_tr_scaled, y_tr, sample_weight=w)

    y_val_pred = final_xgb.predict(X_val_scaled)
    oof_preds[val_idx] = y_val_pred

    mse = mean_squared_error(y_val, y_val_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_val, y_val_pred)
    r2 = r2_score(y_val, y_val_pred)

    fold_metrics.append((mae, rmse, r2))
    fold += 1



In [ ]:
stability_df = pd.DataFrame.from_dict(feature_counts, orient="index", columns=["Frequency"])
stability_df["Frequency"] /= skf.get_n_splits()
consensus_features = stability_df[stability_df["Frequency"] >= 0.6].index.tolist() #use proteins selected in ≥60% of folds

#print(consensus_features)

### Final Model Training and Validation

In [ ]:
X_train_sel = X_train_df[consensus_features].values
X_test_sel = X_test_df[consensus_features].values
X_swat_sel = X_swat[consensus_features].values

scaler_full = StandardScaler()
X_train_scaled = scaler_full.fit_transform(X_train_sel)
X_test_scaled = scaler_full.transform(X_test_sel)
X_swat_scaled = scaler_full.transform(X_swat_sel)

w_full = np.where(y_train >= 25, 4,
          np.where(y_train >= 16, 2, 1))

final_model = make_final_xgb()
final_model.fit(X_train_scaled, y_train, sample_weight=w_full)